# Chapter 4: Preprocessing — Getting Data Model-Ready

### 4.1 Scaling Numeric Features

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_train)     # learn mean/std from TRAINING data only
X_test_scaled = scaler.transform(X_test)      # apply the SAME transformation to test data

⚠️ **This `fit`-on-train, `transform`-on-test pattern is the single most important discipline in this entire volume, and it's exactly what Chapter 5's Pipeline machinery exists to enforce automatically.** If you instead fit the scaler on the *entire* dataset (train + test combined) before splitting, information about the test set's distribution leaks into the training process — a subtle but real form of **data leakage** that makes your evaluation metrics overly optimistic and unreliable once the model meets genuinely new data in production. Always fit preprocessing steps exclusively on training data.

`StandardScaler` isn't the only option: `MinMaxScaler` rescales to a fixed [0,1] range (useful when a model expects bounded inputs); `RobustScaler` uses the median and IQR instead of mean and standard deviation, making it far less sensitive to outliers than `StandardScaler` — worth reaching for whenever your data has the kind of skew and extreme values discussed in Volume 04.

### 4.2 Encoding Categorical Features

In [ ]:
# One-hot encoding: turns a category column into multiple 0/1 columns, one per category
region_encoded = pd.get_dummies(churn_df.assign(Region=["North","South"]*100)["Region"], prefix="Region")
print(region_encoded.head())

Most scikit-learn models (with the notable exception of some tree-based ones in Volume 06) cannot accept raw text categories directly — `"North"`, `"South"` need to become numbers first. **One-hot encoding** is the standard, safe default: each category becomes its own binary column, avoiding the trap of accidentally implying a false numeric order (encoding North=1, South=2, East=3 would wrongly suggest East is somehow "more" than North in a way a linear model might pick up on).

### Cheat Sheet — Preprocessing

| Task | Code |
|---|---|
| Standardize (mean 0, std 1) | `StandardScaler()` |
| Scale to [0,1] | `MinMaxScaler()` |
| Robust to outliers | `RobustScaler()` |
| One-hot encode | `pd.get_dummies(df["col"])` or `OneHotEncoder()` |
| Golden rule | `.fit()` only on training data, `.transform()` on both |

---